# TravelTide Perk Assignment Rule Evaluation

version 2

## Description  
Perk Assignment Rule Evaluation  
**Source:** `tt_project.user_features_transformed` via local Postgres (CLG-NAS)  
**Purpose:** Evaluate field-based threshold rules for each priority perk assignment (P3–P17)  

Each priority section follows the same 7-step process:  
1. Originating cluster dimension and its feature set  
2. Profile target segment on dimension fields only  
3. MIN / P25 / P50 / P75 / MAX per field  
4. Identify clean boundary between target and all other users  
5. Write the candidate rule  
6. Validate: % of target segment captured  
7. Validate: % of non-target users incorrectly captured (false positive rate)  
  
**Success criteria:**  Capture rate ≥ 75% | False positive rate ≤ 15%  
  
**Note:** P1 (never-booked → Exclusive Discounts) and P2 (cancelled-all → No Change Fees) are pre-solved rule-based overrides — not evaluated here.  

## Setup - Connection and Helper Functions

In [4]:
# --- Environment Check & Standard Imports ---
# Run first to confirm environment before any other cells.

import sys
print(f"Python: {sys.version}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlalchemy as sa
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import psycopg2

print(f"NumPy:     {np.__version__}")
print(f"Pandas:    {pd.__version__}")

import psycopg2


Python: 3.14.3 (tags/v3.14.3:323c59a, Feb  3 2026, 16:04:56) [MSC v.1944 64 bit (AMD64)]
NumPy:     2.4.4
Pandas:    3.0.2


In [5]:
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline
sns.style='whitegrid'

# DB connections
load_dotenv()
engine  = create_engine(os.getenv("LOCAL_traveltide_URL"))
con = engine.connect().execution_options(
                    isolation_level="AUTOCOMMIT")
DB_SCHEMA = 'tt_project'
DB_TABLE = 'user_features_transformed'
result = con.execute(text(f'SELECT COUNT(*) FROM {DB_SCHEMA}.{DB_TABLE}'))
row_count = result.scalar()
print(f'Connected. {DB_SCHEMA}.{DB_TABLE} has {row_count:,} rows.')
print("\nEnvironment ready ✓")

Connected. tt_project.user_features_transformed has 5,998 rows.

Environment ready ✓


In [6]:
# ── Load full table into dataframe ─────────────────────────────────────────
# Pulling all columns — full field set available for rule development
query = f'SELECT * FROM {DB_SCHEMA}.{DB_TABLE}'
df = pd.read_sql(query, engine)
df_users = pd.read_sql(query, engine)

print(f'Loaded {len(df):,} users | {df.shape[1]} columns')
print('\nBooking status breakdown:')
print(df['user_booking_status'].value_counts())
print('\nSegment columns present:')
seg_cols = [c for c in df.columns if c.startswith('segment_')]
print(seg_cols)

Loaded 5,998 users | 160 columns

Booking status breakdown:
user_booking_status
booked           5476
never_booked      434
cancelled_all      88
Name: count, dtype: int64

Segment columns present:
['segment_A_website', 'segment_A_spending', 'segment_A_travel', 'segment_A_cancellation', 'segment_B_spending_distance', 'segment_B_cancellation', 'segment_B_hotel_behavior', 'segment_B_flight_discounts', 'segment_B_intl_travel']


In [7]:
# ── Helper: profile dimension fields for two populations ──────────────────
def profile_fields(target_df, other_df, fields,
                   target_label='TARGET', other_label='OTHERS'):
    """Side-by-side percentile table for each field."""
    pcts       = [0, 0.25, 0.50, 0.75, 1.0]
    pct_labels = ['min', 'p25', 'p50', 'p75', 'max']
    rows = []
    for f in fields:
        if f not in df.columns:
            print(f'  ⚠️  Field not found in dataframe: {f}')
            continue
        t_vals = target_df[f].dropna()
        o_vals = other_df[f].dropna()
        row = {'field': f,
               f'{target_label}_n_nonull': len(t_vals),
               f'{other_label}_n_nonull': len(o_vals)}
        for i, lbl in enumerate(pct_labels):
            row[f'{target_label}_{lbl}'] = round(t_vals.quantile(pcts[i]), 4) if len(t_vals) else None
            row[f'{other_label}_{lbl}'] = round(o_vals.quantile(pcts[i]), 4) if len(o_vals) else None
        rows.append(row)
    result = pd.DataFrame(rows).set_index('field')
    # Reorder columns for readability
    t_cols = [c for c in result.columns if c.startswith(target_label)]
    o_cols = [c for c in result.columns if c.startswith(other_label)]
    return result[t_cols + o_cols]


# ── Helper: evaluate a boolean rule mask ──────────────────────────────────
def evaluate_rule(df, rule_mask, target_mask, label):
    """Print capture rate and false positive rate."""
    n_target  = int(target_mask.sum())
    n_others  = int((~target_mask).sum())
    n_cap     = int((rule_mask & target_mask).sum())
    n_fp      = int((rule_mask & ~target_mask).sum())
    n_flagged = int(rule_mask.sum())

    cap_rate  = n_cap / n_target * 100  if n_target > 0 else 0
    fp_rate   = n_fp  / n_others * 100  if n_others > 0 else 0

    ok_cap = '✅' if cap_rate >= 75 else '❌'
    ok_fp  = '✅' if fp_rate  <= 15 else '❌'

    print(f"{'='*65}")
    print(f"  {label}")
    print(f"{'='*65}")
    print(f"  Target segment :  {n_target:,} users")
    print(f"  Flagged by rule:  {n_flagged:,} users")
    print(f"  {ok_cap} Capture rate  : {n_cap:,} / {n_target:,} = {cap_rate:.1f}%   (goal ≥ 75%)")
    print(f"  {ok_fp} False pos rate: {n_fp:,} / {n_others:,} = {fp_rate:.1f}%   (goal ≤ 15%)")
    print()
    return dict(label=label, n_target=n_target, n_flagged=n_flagged,
                capture_pct=round(cap_rate,1), fp_pct=round(fp_rate,1))


# ── Helper: compare multiple candidate rules in one call ──────────────────
def compare_candidates(df, candidates_dict, target_mask, priority):
    """Run evaluate_rule on every candidate and return summary DataFrame."""
    results = []
    for label, mask in candidates_dict.items():
        r = evaluate_rule(df, mask, target_mask, f'{priority} | {label}')
        results.append(r)
    return pd.DataFrame(results).set_index('label')


print('Helper functions loaded.')

Helper functions loaded.


## Overall Perk assignment assessments

#### Perk-Level Combined Signal Profiling
Profile all segments belonging to each perk as a single combined population.
Goal: find the minimum field conditions that capture the full perk population
without needing individual segment-level rules.

#### Groupings

Premium Travel Benefits:   
1. highvolume_package_travelers,  
2. flight_only_high_fare,  
3. frequent_mixed_travelers  
4. Fallback >= 3000 completed_total_gross  
 Free Checked Bags:  
1. dedicated_intl_travelers,  
2. pct_complete_flights_intl > 0.95,  
Free Hotel Meals:  
1. luxury_hotel_discount_seekers,  
2. hotel_only_domestic_stays,  
3. hotel_centric_short_haul,  
4. regular_hotel_bookers,  
Exclusive Discounts:  
1. heavy_flight_discount_users,  
2. never_booked  
3. Fallback <= 3000 completed_total_gross  
4. Fallback no other primary signal  
No Change Fees:  
1. highvalue_advanced_cancellers,  
2. regular_hotel_bookers,  
3. flight_advanced_planner_cancellers,  
4. package_cancellers  
5. intl_trip_cancellers  
6. hotel_lastminute_cancellers  

## Perk-level combined signal profiling

In [ ]:
# ── Perk-Level Combined Signal Profiling ─────────────────────────────────
# For each perk, pool ALL its qualifying segment signals into one combined mask.
# Then profile that population on its most diagnostic behavioral fields.
# Goal: identify whether a small set of thresholds can replace the per-segment
#       priority chain and still capture the same users.

# ── Define combined perk populations from segment labels ──────────────────
perk_segments = {
    'Premium Travel Benefits': {
        'segment_B_flight_discounts': ['highvolume_package_travelers'],
        'segment_A_spending':         ['flight_only_high_fare'],
        'segment_B_intl_travel':      ['frequent_mixed_travelers'],
    },
    'Free Checked Bags': {
        'segment_B_intl_travel': ['dedicated_intl_travelers'],
        # P6 field-based override included below via pct_complete_flights_intl
    },
    'Free Hotel Meals': {
        'segment_B_hotel_behavior': ['luxury_hotel_discount_seekers',
                                     'regular_hotel_bookers'],
        'segment_A_spending':       ['hotel_only_domestic_stays'],
        'segment_A_travel':         ['hotel_centric_short_haul'],
    },
    'Exclusive Discounts': {
        'segment_B_flight_discounts': ['heavy_flight_discount_users'],
        # never_booked override and fallbacks included below
    },
    'No Change Fees': {
        'segment_A_cancellation': ['highvalue_advanced_cancellers',
                                   'flight_advanced_planner_cancellers',
                                   'package_cancellers',
                                   'hotel_lastminute_cancellers'],
        'segment_B_intl_travel':  ['intl_trip_cancellers'],
        # cancelled_all override included below
    },
}

# ── Additional override/fallback masks ────────────────────────────────────
fcb_p6_override  = df_users['pct_complete_flights_intl'] > 0.90
excl_never_booked = df_users['user_booking_status'] == 'never_booked'
ncf_cancelled_all = df_users['user_booking_status'] == 'cancelled_all'
ptb_fallback      = df_users['completed_total_gross'] >= 3000
excl_fallback     = df_users['completed_total_gross'] < 3000

# ── Build combined boolean mask per perk ──────────────────────────────────
def build_perk_mask(df, seg_dict, extra_masks=None):
    mask = pd.Series(False, index=df.index)
    for col, vals in seg_dict.items():
        for v in vals:
            mask = mask | (df[col] == v)
    if extra_masks:
        for m in extra_masks:
            mask = mask | m
    return mask

perk_masks = {
    'Premium Travel Benefits': build_perk_mask(
        df_users, perk_segments['Premium Travel Benefits']),

    'Free Checked Bags': build_perk_mask(
        df_users, perk_segments['Free Checked Bags'],
        extra_masks=[fcb_p6_override]),

    'Free Hotel Meals': build_perk_mask(
        df_users, perk_segments['Free Hotel Meals']),

    'Exclusive Discounts': build_perk_mask(
        df_users, perk_segments['Exclusive Discounts'],
        extra_masks=[excl_never_booked]),

    'No Change Fees': build_perk_mask(
        df_users, perk_segments['No Change Fees'],
        extra_masks=[ncf_cancelled_all]),
}

# ── Diagnostic fields per perk ────────────────────────────────────────────
perk_fields = {
    'Premium Travel Benefits': [
        'completed_total_gross', 'trips_per_year', 'total_package_bookings',
        'flight_discount_rate', 'avg_fare_per_seat', 'pct_complete_flights_intl',
        'completed_avg_km', 'total_hotel_only_bookings',
    ],
    'Free Checked Bags': [
        'pct_complete_flights_intl', 'completed_avg_km', 'avg_checked_bags',
        'total_cancellation_rate', 'avg_fare_per_seat', 'return_flight_rate',
        'total_package_bookings',
    ],
    'Free Hotel Meals': [
        'avg_trip_nights', 'total_hotel_only_bookings', 'hotel_discount_rate',
        'avg_hotel_per_room', 'completed_hotel_gross', 'avg_hotel_discount_amt',
        'total_package_bookings', 'completed_avg_km',
    ],
    'Exclusive Discounts': [
        'flight_discount_rate', 'completed_total_gross', 'trips_per_year',
        'booking_conversion_rate', 'total_package_bookings',
        'avg_flight_discount_amt', 'life_total_trips',
    ],
    'No Change Fees': [
        'total_cancellation_rate', 'cancel_spend_ratio',
        'flight_only_cxl_rate', 'hotel_only_cxl_rate', 'package_cxl_rate',
        'avg_cancelled_lead_time', 'pct_cxl_flights_intl',
        'cxl_gross_spend',
    ],
}

# ── Profile each perk: combined population vs all others ──────────────────
pcts       = [0, 0.25, 0.50, 0.75, 1.0]
pct_labels = ['min', 'p25', 'p50', 'p75', 'max']

for perk, mask in perk_masks.items():
    target = df_users[mask]
    others = df_users[~mask]
    fields = perk_fields[perk]

    print(f"\n{'='*80}")
    print(f"  PERK: {perk}")
    print(f"  Combined population: {mask.sum():,} users "
          f"({mask.sum()/len(df_users)*100:.1f}% of cohort)")
    print(f"  Others: {(~mask).sum():,} users")
    print(f"{'='*80}")
    print(f"\n  {'Field':<35} "
          f"{'T_min':>8} {'T_p25':>8} {'T_p50':>8} {'T_p75':>8} {'T_max':>8}  "
          f"{'O_min':>8} {'O_p25':>8} {'O_p50':>8} {'O_p75':>8} {'O_max':>8}")
    print(f"  {'-'*115}")

    for f in fields:
        if f not in df_users.columns:
            print(f"  ⚠️  {f} not found")
            continue
        t = target[f].dropna()
        o = others[f].dropna()
        if len(t) == 0:
            continue
        t_vals = [t.quantile(p) for p in pcts]
        o_vals = [o.quantile(p) for p in pcts]
        print(f"  {f:<35} "
              f"{t_vals[0]:>8.3f} {t_vals[1]:>8.3f} {t_vals[2]:>8.3f} "
              f"{t_vals[3]:>8.3f} {t_vals[4]:>8.3f}  "
              f"{o_vals[0]:>8.3f} {o_vals[1]:>8.3f} {o_vals[2]:>8.3f} "
              f"{o_vals[3]:>8.3f} {o_vals[4]:>8.3f}")
    print()


  PERK: Premium Travel Benefits
  Combined population: 2,663 users (44.4% of cohort)
  Others: 3,335 users

  Field                                  T_min    T_p25    T_p50    T_p75    T_max     O_min    O_p25    O_p50    O_p75    O_max
  -------------------------------------------------------------------------------------------------------------------
  completed_total_gross                  9.270 2325.685 3545.120 5272.770 24382.450    84.000 1157.140 2117.320 3732.195 28364.430
  trips_per_year                         0.770    6.450    8.540   10.900   26.730     0.000    2.280    4.450    7.020   26.090
  total_package_bookings                 0.000    2.000    3.000    4.000    7.000     0.000    0.000    1.000    2.000    8.000
  flight_discount_rate                   0.000    0.000    0.167    0.333    1.000     0.000    0.000    0.000    0.000    1.000
  avg_fare_per_seat                      9.270  268.131  348.822  454.783 2526.600     5.350  229.651  350.957  482.069 3026.4

In [9]:
# ── Single-rule capture vs false-positive test per perk ───────────────────
# For each perk, test whether a 1–2 field rule can approximate the full
# combined segment population. Baseline for deciding if the priority chain
# can be simplified.

def test_rule(label, rule_mask, target_mask, df):
    n_t   = target_mask.sum()
    n_o   = (~target_mask).sum()
    n_cap = (rule_mask & target_mask).sum()
    n_fp  = (rule_mask & ~target_mask).sum()
    cap   = n_cap / n_t * 100 if n_t > 0 else 0
    fp    = n_fp  / n_o * 100 if n_o > 0 else 0
    ok_c  = '✅' if cap >= 75 else '❌'
    ok_f  = '✅' if fp  <= 15 else '❌'
    print(f"  {label}")
    print(f"    {ok_c} Capture: {n_cap:,}/{n_t:,} = {cap:.1f}%   "
          f"{ok_f} FP rate: {n_fp:,}/{n_o:,} = {fp:.1f}%   "
          f"Flagged: {rule_mask.sum():,}")
    print()

print("=== Single-Rule Approximation — Can One Rule Replace the Stack? ===\n")
print("Criteria: Capture ≥ 75% of combined target | False positive ≤ 15%\n")

# ── Premium Travel Benefits ───────────────────────────────────────────────
print("── Premium Travel Benefits ──────────────────────────────────────────")
ptb_mask = perk_masks['Premium Travel Benefits']

test_rule("total_package_bookings >= 2",
          df_users['total_package_bookings'] >= 2, ptb_mask, df_users)

test_rule("total_package_bookings >= 2 AND trips_per_year >= 2",
          (df_users['total_package_bookings'] >= 2) &
          (df_users['trips_per_year'] >= 2), ptb_mask, df_users)

test_rule("completed_total_gross >= 2500 AND total_package_bookings >= 1",
          (df_users['completed_total_gross'] >= 2500) &
          (df_users['total_package_bookings'] >= 1), ptb_mask, df_users)

# ── Free Checked Bags ─────────────────────────────────────────────────────
print("── Free Checked Bags ────────────────────────────────────────────────")
fcb_mask = perk_masks['Free Checked Bags']

test_rule("pct_complete_flights_intl >= 0.80",
          df_users['pct_complete_flights_intl'] >= 0.80, fcb_mask, df_users)

test_rule("pct_complete_flights_intl >= 0.90",
          df_users['pct_complete_flights_intl'] >= 0.90, fcb_mask, df_users)

test_rule("pct_complete_flights_intl >= 0.95",
          df_users['pct_complete_flights_intl'] >= 0.95, fcb_mask, df_users)

# ── Free Hotel Meals ──────────────────────────────────────────────────────
print("── Free Hotel Meals ─────────────────────────────────────────────────")
fhm_mask = perk_masks['Free Hotel Meals']

test_rule("total_hotel_only_bookings >= 1",
          df_users['total_hotel_only_bookings'] >= 1, fhm_mask, df_users)

test_rule("total_hotel_only_bookings >= 1 AND avg_trip_nights >= 3",
          (df_users['total_hotel_only_bookings'] >= 1) &
          (df_users['avg_trip_nights'] >= 3), fhm_mask, df_users)

test_rule("total_hotel_only_bookings >= 1 AND avg_trip_nights >= 4",
          (df_users['total_hotel_only_bookings'] >= 1) &
          (df_users['avg_trip_nights'] >= 4), fhm_mask, df_users)

test_rule("hotel_discount_rate >= 0.30 OR (total_hotel_only_bookings >= 1 AND avg_trip_nights >= 4)",
          (df_users['hotel_discount_rate'] >= 0.30) |
          ((df_users['total_hotel_only_bookings'] >= 1) &
           (df_users['avg_trip_nights'] >= 4)), fhm_mask, df_users)

# ── Exclusive Discounts ───────────────────────────────────────────────────
print("── Exclusive Discounts ──────────────────────────────────────────────")
excl_mask = perk_masks['Exclusive Discounts']

test_rule("flight_discount_rate >= 0.50 OR never_booked",
          (df_users['flight_discount_rate'] >= 0.50) |
          (df_users['user_booking_status'] == 'never_booked'), excl_mask, df_users)

test_rule("flight_discount_rate >= 0.33 OR never_booked",
          (df_users['flight_discount_rate'] >= 0.33) |
          (df_users['user_booking_status'] == 'never_booked'), excl_mask, df_users)

test_rule("life_total_trips <= 1 OR flight_discount_rate >= 0.50",
          (df_users['life_total_trips'] <= 1) |
          (df_users['flight_discount_rate'] >= 0.50), excl_mask, df_users)

# ── No Change Fees ────────────────────────────────────────────────────────
print("── No Change Fees ───────────────────────────────────────────────────")
ncf_mask = perk_masks['No Change Fees']

test_rule("total_cancellation_rate >= 0.20",
          df_users['total_cancellation_rate'] >= 0.20, ncf_mask, df_users)

test_rule("total_cancellation_rate >= 0.25 OR cancelled_all",
          (df_users['total_cancellation_rate'] >= 0.25) |
          (df_users['user_booking_status'] == 'cancelled_all'), ncf_mask, df_users)

test_rule("cancel_spend_ratio >= 0.25 OR cancelled_all",
          (df_users['cancel_spend_ratio'] >= 0.25) |
          (df_users['user_booking_status'] == 'cancelled_all'), ncf_mask, df_users)

test_rule("total_cancellation_rate >= 0.20 AND cancel_spend_ratio >= 0.10",
          (df_users['total_cancellation_rate'] >= 0.20) &
          (df_users['cancel_spend_ratio'] >= 0.10), ncf_mask, df_users)

=== Single-Rule Approximation — Can One Rule Replace the Stack? ===

Criteria: Capture ≥ 75% of combined target | False positive ≤ 15%

── Premium Travel Benefits ──────────────────────────────────────────
  total_package_bookings >= 2
    ✅ Capture: 2,371/2,663 = 89.0%   ❌ FP rate: 1,214/3,335 = 36.4%   Flagged: 3,585

  total_package_bookings >= 2 AND trips_per_year >= 2
    ✅ Capture: 2,370/2,663 = 89.0%   ❌ FP rate: 1,206/3,335 = 36.2%   Flagged: 3,576

  completed_total_gross >= 2500 AND total_package_bookings >= 1
    ❌ Capture: 1,887/2,663 = 70.9%   ❌ FP rate: 1,025/3,335 = 30.7%   Flagged: 2,912

── Free Checked Bags ────────────────────────────────────────────────
  pct_complete_flights_intl >= 0.80
    ✅ Capture: 964/1,002 = 96.2%   ✅ FP rate: 29/4,996 = 0.6%   Flagged: 993

  pct_complete_flights_intl >= 0.90
    ✅ Capture: 959/1,002 = 95.7%   ✅ FP rate: 0/4,996 = 0.0%   Flagged: 959

  pct_complete_flights_intl >= 0.95
    ✅ Capture: 959/1,002 = 95.7%   ✅ FP rate: 0/4,996 =

### P3 - Premium Travel Benefits  
**Source dimension:** Path B · flight_discounts    
**Target segment:** `highvolume_package_travelers` (n≈1,262)  
**Dimension fields:** `flight_discount_rate`, `avg_flight_discount_pct`, `avg_flight_discount_amt`,
`total_flight_disc`, `overall_discount_pct`, `discount_dependency`,
`total_flight_only_bookings`, `total_package_bookings`  
**Key signals:** Highest total gross ($4,458), most trips/yr (3.9), most package bookings (3.1), moderate discount rate (0.36)

In [10]:
# Steps 1–3: Define target mask and profile dimension fields
P3_DIM_FIELDS = [
    'overall_discount_pct', # key
    'total_package_bookings', # key
    # Supporting
    'trips_per_year', # key
    'completed_total_gross', # key
    'flight_discount_rate',
    'avg_flight_discount_pct',
    'avg_flight_discount_amt',
    'total_flight_disc',
    'discount_dependency',
    'total_flight_only_bookings'
]

p3_target = df['segment_B_flight_discounts'] == 'highvolume_package_travelers'
print(f'P3 target segment: {p3_target.sum():,} users')
print()
profile_fields(df[p3_target], df[~p3_target], P3_DIM_FIELDS)

P3 target segment: 1,262 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
overall_discount_pct,1262,0.0001,0.0078,0.0163,0.0297,0.1182,3783,0.0000,0.0000,0.0000,0.0078,0.3500
total_package_bookings,1262,0.0000,2.0000,3.0000,4.0000,7.0000,4736,0.0000,1.0000,2.0000,3.0000,8.0000
trips_per_year,1262,2.2500,7.0200,8.9800,11.4100,25.7200,4736,0.0000,2.8500,5.6500,8.3600,26.7300
completed_total_gross,1262,277.4900,2616.0375,3850.9450,5633.8375,24221.0700,4180,9.2700,1381.3925,2490.9350,4157.5500,28364.4300
flight_discount_rate,1262,0.1429,0.2500,0.3333,0.5000,1.0000,3783,0.0000,0.0000,0.0000,0.0000,1.0000
avg_flight_discount_pct,1262,0.0004,0.0180,0.0347,0.0588,0.1887,3783,0.0000,0.0000,0.0000,0.0000,0.4500
avg_flight_discount_amt,1262,0.1134,6.5264,13.2079,23.6785,166.2666,3918,0.0000,0.0000,0.0000,0.0000,2947.5900
total_flight_disc,1262,0.4535,21.5844,41.7780,74.8830,389.1420,3783,0.0000,0.0000,0.0000,0.0000,2564.5360
discount_dependency,1262,0.0000,0.0000,0.0000,0.0000,0.5000,4280,0.0000,0.0000,0.0000,0.0000,1.0000


In [11]:
# Steps 4–7: Test candidate rules
# Key insight: flight_discount_rate min=0.14 (all targets have ≥1 discounted flight)
#              trips_per_year p10=5.6 for target vs near-zero for non-bookers
#              total_package_bookings p25=2 for target

p3_candidates = {
    'pkg>=2 AND trips>=5 AND fdr>=0.14': (
        (df['total_package_bookings'] >= 2) &
        (df['trips_per_year'] >= 5) &
        (df['flight_discount_rate'] >= 0.14)
    ),
    'pkg>=2 AND trips>=5 AND fdr>=0.20': (
        (df['total_package_bookings'] >= 2) &
        (df['trips_per_year'] >= 5) &
        (df['flight_discount_rate'] >= 0.20)
    ),
    'pkg>=3 AND trips>=5 AND fdr>=0.14': (
        (df['total_package_bookings'] >= 3) &
        (df['trips_per_year'] >= 5) &
        (df['flight_discount_rate'] >= 0.14)
    ),
    'pkg>=1 AND trips>=2.25 AND fdr>=0.14 AND ttfd>=0.45': (
        (df['total_package_bookings'] >= 2) &
        (df['trips_per_year'] >= 3) &
        (df['flight_discount_rate'] >= 0.15) & 
        (df['avg_flight_discount_pct'] <= 0.20) &
        (df['overall_discount_pct'] <= 0.10)        
    ),
}

summary_p3 = compare_candidates(df, p3_candidates, p3_target, 'P3')
summary_p3

  P3 | pkg>=2 AND trips>=5 AND fdr>=0.14
  Target segment :  1,262 users
  Flagged by rule:  1,228 users
  ✅ Capture rate  : 1,123 / 1,262 = 89.0%   (goal ≥ 75%)
  ✅ False pos rate: 105 / 4,736 = 2.2%   (goal ≤ 15%)

  P3 | pkg>=2 AND trips>=5 AND fdr>=0.20
  Target segment :  1,262 users
  Flagged by rule:  1,186 users
  ✅ Capture rate  : 1,081 / 1,262 = 85.7%   (goal ≥ 75%)
  ✅ False pos rate: 105 / 4,736 = 2.2%   (goal ≤ 15%)

  P3 | pkg>=3 AND trips>=5 AND fdr>=0.14
  Target segment :  1,262 users
  Flagged by rule:  876 users
  ❌ Capture rate  : 836 / 1,262 = 66.2%   (goal ≥ 75%)
  ✅ False pos rate: 40 / 4,736 = 0.8%   (goal ≤ 15%)

  P3 | pkg>=1 AND trips>=2.25 AND fdr>=0.14 AND ttfd>=0.45
  Target segment :  1,262 users
  Flagged by rule:  1,300 users
  ✅ Capture rate  : 1,185 / 1,262 = 93.9%   (goal ≥ 75%)
  ✅ False pos rate: 115 / 4,736 = 2.4%   (goal ≤ 15%)



,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P3 | pkg>=2 AND trips>=5 AND fdr>=0.14,1262,1228,89.0000,2.2000
P3 | pkg>=2 AND trips>=5 AND fdr>=0.20,1262,1186,85.7000,2.2000
P3 | pkg>=3 AND trips>=5 AND fdr>=0.14,1262,876,66.2000,0.8000
P3 | pkg>=1 AND trips>=2.25 AND fdr>=0.14 AND ttfd>=0.45,1262,1300,93.9000,2.4000


In [12]:
# ── ACCEPTED P3 RULE — update thresholds after reviewing candidates above ──
P3_RULE = (
        (df['total_package_bookings'] >= 2) &
        (df['trips_per_year'] >= 3) &
        (df['flight_discount_rate'] >= 0.15) & 
        (df['avg_flight_discount_pct'] <= 0.20) &
        (df['overall_discount_pct'] <= 0.10) 
)
# SQL:
#   total_package_bookings >= 2
#   AND trips_per_year >= 5
#   AND flight_discount_rate >= 0.14
#   → Premium Travel Benefits

evaluate_rule(df, P3_RULE, p3_target, 'P3 — highvolume_package_travelers → Premium Travel Benefits')

  P3 — highvolume_package_travelers → Premium Travel Benefits
  Target segment :  1,262 users
  Flagged by rule:  1,300 users
  ✅ Capture rate  : 1,185 / 1,262 = 93.9%   (goal ≥ 75%)
  ✅ False pos rate: 115 / 4,736 = 2.4%   (goal ≤ 15%)



{'label': 'P3 — highvolume_package_travelers → Premium Travel Benefits',
 'n_target': 1262,
 'n_flagged': 1300,
 'capture_pct': 93.9,
 'fp_pct': 2.4}

### P4 - Premium Travel Benefits  
**Source dimension:** Path A · spending - cluster 2  
**Target segment:** `flight_only_high_fare` (n≈107)  
**Dimension fields:** `log_avg_fare_per_seat`, `log_avg_hotel_per_room`, `log_avg_hotel_total_per_stay`,
`log_completed_total_gross`, `log_completed_flight_gross`, `log_completed_hotel_gross`,
`log_attempted_total_gross`  
**Key signals:** Zero hotel bookings, zero hotel gross, avg_trip_nights=1.0 (constant), total_hotel_only=0, total_package=0

In [13]:
P4_DIM_FIELDS = [
    # 'log_avg_fare_per_seat',
    # 'log_avg_hotel_per_room',
    # 'log_avg_hotel_total_per_stay',
    # 'log_completed_total_gross',
    'completed_total_gross',
    # 'log_completed_flight_gross',
    'completed_flight_gross',
    # 'log_completed_hotel_gross',
    # 'log_attempted_total_gross',
    'attempted_total_gross',
    # Raw supporting fields for rule writing
    'avg_fare_per_seat',
    'avg_hotel_per_room',
    'avg_trip_nights',
    'total_package_bookings',
    'total_hotel_only_bookings',
    'total_flight_only_bookings',
]

p4_target = df['segment_A_spending'] == 'flight_only_high_fare'
print(f'P4 target segment: {p4_target.sum():,} users')
print()
profile_fields(df[p4_target], df[~p4_target], P4_DIM_FIELDS)

P4 target segment: 107 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
completed_total_gross,107,9.2700,237.5300,421.4400,711.8400,5053.2000,5335,84.0000,1684.5700,2865.0000,4617.0350,28364.4300
completed_flight_gross,107,9.2700,237.5300,421.4400,711.8400,5053.2000,4938,5.3500,566.3000,968.5100,1517.8475,22234.0700
attempted_total_gross,107,9.2700,248.7950,509.5700,1191.7050,6330.9400,5435,84.0000,1760.5300,3037.6100,4859.9900,28364.4300
avg_fare_per_seat,107,9.2700,201.0400,384.9200,563.6000,2526.6000,4938,5.3500,252.5790,349.3692,465.0475,3026.4500
avg_hotel_per_room,1,39.0000,39.0000,39.0000,39.0000,39.0000,5335,21.0000,124.0000,163.5000,212.4500,1063.0000
avg_trip_nights,1,1.0000,1.0000,1.0000,1.0000,1.0000,5335,1.0000,2.7500,4.0000,5.3333,30.0000
total_package_bookings,107,0.0000,0.0000,0.0000,0.0000,1.0000,5891,0.0000,1.0000,2.0000,3.0000,8.0000
total_hotel_only_bookings,107,0.0000,0.0000,0.0000,0.0000,1.0000,5891,0.0000,0.0000,0.0000,1.0000,3.0000
total_flight_only_bookings,107,0.0000,1.0000,1.0000,1.0000,2.0000,5891,0.0000,0.0000,0.0000,0.0000,3.0000


In [14]:
# Key insight: hotel bookings = 0 AND package = 0 defines this segment cleanly
# avg_hotel_per_room will be 0/NULL for all these users (no hotel spend)
# avg_fare_per_seat distinguishes from hotel-only users

p4_candidates = {
    'hotel_only=0 AND pkg=0 AND hotel_room=0': (
        (df['total_hotel_only_bookings'] == 0) &
        (df['avg_fare_per_seat'] >= 150) &
        (df['total_package_bookings'] == 0) &
        (df['total_flight_only_bookings'] >= 1) #&
        # (df['avg_hotel_per_room'] <= 50)
    ),
    'hotel_only=0 AND pkg=0 AND fare>=200': (
        (df['total_hotel_only_bookings'] == 0) &
        (df['total_package_bookings'] == 0) &
        (df['avg_fare_per_seat'] >= 150)
    ),
    'hotel_only=0 AND pkg=0 AND fare>=300': (
        (df['total_hotel_only_bookings'] == 0) &
        (df['total_package_bookings'] == 0) &
        (df['avg_fare_per_seat'] >= 300)
    ),
    'hotel_only=0 AND pkg=0 AND nights<=2': (
        (df['total_hotel_only_bookings'] == 0) &
        (df['total_package_bookings'] == 0) &
        (df['avg_trip_nights'] <= 2)
    ),
}

summary_p4 = compare_candidates(df, p4_candidates, p4_target, 'P4')
summary_p4

  P4 | hotel_only=0 AND pkg=0 AND hotel_room=0
  Target segment :  107 users
  Flagged by rule:  79 users
  ❌ Capture rate  : 79 / 107 = 73.8%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,891 = 0.0%   (goal ≤ 15%)

  P4 | hotel_only=0 AND pkg=0 AND fare>=200
  Target segment :  107 users
  Flagged by rule:  79 users
  ❌ Capture rate  : 79 / 107 = 73.8%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,891 = 0.0%   (goal ≤ 15%)

  P4 | hotel_only=0 AND pkg=0 AND fare>=300
  Target segment :  107 users
  Flagged by rule:  56 users
  ❌ Capture rate  : 56 / 107 = 52.3%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,891 = 0.0%   (goal ≤ 15%)

  P4 | hotel_only=0 AND pkg=0 AND nights<=2
  Target segment :  107 users
  Flagged by rule:  0 users
  ❌ Capture rate  : 0 / 107 = 0.0%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,891 = 0.0%   (goal ≤ 15%)



,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P4 | hotel_only=0 AND pkg=0 AND hotel_room=0,107,79,73.8000,0.0000
P4 | hotel_only=0 AND pkg=0 AND fare>=200,107,79,73.8000,0.0000
P4 | hotel_only=0 AND pkg=0 AND fare>=300,107,56,52.3000,0.0000
P4 | hotel_only=0 AND pkg=0 AND nights<=2,107,0,0.0000,0.0000


In [15]:
# ── ACCEPTED P4 RULE ───────────────────────────────────────────────────────
P4_RULE = (
    (df['total_hotel_only_bookings'] == 0) &
    (df['total_package_bookings'] == 0) &
    (df['avg_hotel_per_room'] == 0)
)
# SQL:
#   total_hotel_only_bookings = 0
#   AND total_package_bookings = 0
#   AND avg_hotel_per_room = 0   (or IS NULL depending on your NULL handling)
#   → Premium Travel Benefits

evaluate_rule(df, P4_RULE, p4_target, 'P4 — flight_only_high_fare → Premium Travel Benefits')

  P4 — flight_only_high_fare → Premium Travel Benefits
  Target segment :  107 users
  Flagged by rule:  0 users
  ❌ Capture rate  : 0 / 107 = 0.0%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,891 = 0.0%   (goal ≤ 15%)



{'label': 'P4 — flight_only_high_fare → Premium Travel Benefits',
 'n_target': 107,
 'n_flagged': 0,
 'capture_pct': 0.0,
 'fp_pct': 0.0}

### P5 / P6 - Free Checked Bags  
**Source dimension:** Path B · intl_travel  
**Target segment:** `dedicated_intl_travelers` (n≈931)  
**Dimension fields:** `pct_complete_flights_intl`, `pct_complete_flights_domestic`,
`log_completed_avg_km`, `log_attempted_avg_km`, `cxl_avg_km`,
`pct_cxl_flights_intl`, `avg_checked_bags`, `avg_seats`,
`return_flight_rate`, `total_package_bookings`  
**Key signals:** 99% intl flights, avg_km 2,535, low cancellation rate 0.013  
**Note:** P6 is the `pct_intl >= 0.95` safety net — P5 and P6 are tested together here.

In [16]:
P5_DIM_FIELDS = [
    'pct_complete_flights_intl',
    'pct_complete_flights_domestic',
    'log_completed_avg_km',
    'log_attempted_avg_km',
    'cxl_avg_km',
    'pct_cxl_flights_intl',
    'avg_checked_bags',
    'avg_seats',
    'return_flight_rate',
    'total_package_bookings',
    # Raw for readability
    'completed_avg_km',
    'total_cancellation_rate',
]

p5_target = df['segment_B_intl_travel'] == 'dedicated_intl_travelers'
print(f'P5 target segment: {p5_target.sum():,} users')
print()
profile_fields(df[p5_target], df[~p5_target], P5_DIM_FIELDS)

P5 target segment: 931 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
pct_complete_flights_intl,931,0.6667,1.0000,1.0000,1.0000,1.0000,4114,0.0000,0.0000,0.2000,0.5000,1.0000
pct_complete_flights_domestic,931,0.0000,0.0000,0.0000,0.0000,0.3333,4114,0.0000,0.5000,0.8000,1.0000,1.0000
log_completed_avg_km,931,5.1749,7.3558,7.7091,7.9901,9.6687,5067,0.0000,6.7867,7.4557,7.7834,9.2254
log_attempted_avg_km,931,5.1749,7.3487,7.7061,7.9855,9.6687,5067,0.0000,6.9486,7.4989,7.8233,9.6764
cxl_avg_km,931,0.0000,0.0000,0.0000,0.0000,4151.7735,5067,0.0000,0.0000,0.0000,0.0000,15935.6599
pct_cxl_flights_intl,931,0.0000,0.0000,0.0000,0.0000,0.5000,5067,0.0000,0.0000,0.0000,0.0000,1.0000
avg_checked_bags,931,0.0000,0.0000,0.5000,1.0000,3.0000,4114,0.0000,0.3333,0.5000,1.0000,5.0000
avg_seats,931,1.0000,1.0000,1.0000,1.0000,4.0000,4114,1.0000,1.0000,1.0000,1.3333,6.0000
return_flight_rate,931,0.0000,0.6000,1.0000,1.0000,1.0000,4511,0.0000,0.6667,1.0000,1.0000,1.0000


In [17]:
p5_candidates = {
    'pct_intl>=0.95': (
        df['pct_complete_flights_intl'] >= 0.95
    ),
    'pct_intl>=0.90 AND avg_km>=1500': (
        (df['pct_complete_flights_intl'] >= 0.90) &
        (df['completed_avg_km'] >= 1500)
    ),
    'pct_intl>=0.95 AND bags>=1': (
        (df['pct_complete_flights_intl'] >= 0.95) &
        (df['avg_checked_bags'] >= 1)
    ),
    'pct_intl>=0.95 AND cxl<=0.10': (
        (df['pct_complete_flights_intl'] >= 0.85) &
        (df['total_cancellation_rate'] <= 0.10) &
        (df['completed_total_km'] >= 2500)
    ),
}

summary_p5 = compare_candidates(df, p5_candidates, p5_target, 'P5')
summary_p5

  P5 | pct_intl>=0.95
  Target segment :  931 users
  Flagged by rule:  959 users
  ✅ Capture rate  : 888 / 931 = 95.4%   (goal ≥ 75%)
  ✅ False pos rate: 71 / 5,067 = 1.4%   (goal ≤ 15%)

  P5 | pct_intl>=0.90 AND avg_km>=1500
  Target segment :  931 users
  Flagged by rule:  739 users
  ❌ Capture rate  : 680 / 931 = 73.0%   (goal ≥ 75%)
  ✅ False pos rate: 59 / 5,067 = 1.2%   (goal ≤ 15%)

  P5 | pct_intl>=0.95 AND bags>=1
  Target segment :  931 users
  Flagged by rule:  349 users
  ❌ Capture rate  : 316 / 931 = 33.9%   (goal ≥ 75%)
  ✅ False pos rate: 33 / 5,067 = 0.7%   (goal ≤ 15%)

  P5 | pct_intl>=0.95 AND cxl<=0.10
  Target segment :  931 users
  Flagged by rule:  602 users
  ❌ Capture rate  : 600 / 931 = 64.4%   (goal ≥ 75%)
  ✅ False pos rate: 2 / 5,067 = 0.0%   (goal ≤ 15%)



,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P5 | pct_intl>=0.95,931,959,95.4000,1.4000
P5 | pct_intl>=0.90 AND avg_km>=1500,931,739,73.0000,1.2000
P5 | pct_intl>=0.95 AND bags>=1,931,349,33.9000,0.7000
P5 | pct_intl>=0.95 AND cxl<=0.10,931,602,64.4000,0.0000


In [18]:
# ── ACCEPTED P5/P6 RULE ────────────────────────────────────────────────────
P5_RULE = (df['pct_complete_flights_intl'] >= 0.95)
# SQL: pct_complete_flights_intl >= 0.95  → Free Checked Bags

evaluate_rule(df, P5_RULE, p5_target, 'P5/P6 — dedicated_intl_travelers → Free Checked Bags')

  P5/P6 — dedicated_intl_travelers → Free Checked Bags
  Target segment :  931 users
  Flagged by rule:  959 users
  ✅ Capture rate  : 888 / 931 = 95.4%   (goal ≥ 75%)
  ✅ False pos rate: 71 / 5,067 = 1.4%   (goal ≤ 15%)



{'label': 'P5/P6 — dedicated_intl_travelers → Free Checked Bags',
 'n_target': 931,
 'n_flagged': 959,
 'capture_pct': 95.4,
 'fp_pct': 1.4}

### P7 - Free Hotel Meals
**Source dimension:** Path B · hotel_behavior  
**Target segment:** `luxury_hotel_discount_seekers` (n≈290)  
**Dimension fields:** `avg_trip_nights`, `log_avg_hotel_total_per_stay`, `log_avg_hotel_per_room`,
`log_completed_hotel_gross`, `hotel_discount_rate`, `avg_hotel_discount_pct`,
`avg_hotel_discount_amt`, `total_hotel_disc`, `total_hotel_only_bookings`,
`log_avg_days_between_trips`  
**Key signals:** hotel_discount_rate 0.638, avg_hotel_discount_amt $160, avg_trip_nights 6.8

In [19]:
P7_DIM_FIELDS = [
    'avg_trip_nights',
    'log_avg_hotel_total_per_stay',
    'log_avg_hotel_per_room',
    'log_completed_hotel_gross',
    'hotel_discount_rate',
    'avg_hotel_discount_pct',
    'avg_hotel_discount_amt',
    'total_hotel_disc',
    'total_hotel_only_bookings',
    # Raw supporting
    'avg_hotel_per_room',
    'completed_hotel_gross',
]

p7_target = df['segment_B_hotel_behavior'] == 'luxury_hotel_discount_seekers'
print(f'P7 target segment: {p7_target.sum():,} users')
print()
profile_fields(df[p7_target], df[~p7_target], P7_DIM_FIELDS)

P7 target segment: 290 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
avg_trip_nights,290,1.0000,4.0417,5.9000,8.0000,30.0000,5046,1.0000,2.6667,3.6667,5.2000,27.0000
log_avg_hotel_total_per_stay,290,4.1431,6.6132,7.1458,7.5744,9.3211,5708,0.0000,5.8214,6.4203,6.9075,9.5546
log_avg_hotel_per_room,290,3.4657,4.8442,5.1279,5.4362,6.5453,5708,0.0000,4.6728,5.0370,5.3230,6.9698
log_completed_hotel_gross,290,4.1431,7.3000,7.9694,8.4948,10.0142,5708,0.0000,6.4085,7.3182,7.9506,10.2477
hotel_discount_rate,290,0.2000,0.4000,0.5000,1.0000,1.0000,5046,0.0000,0.0000,0.0000,0.2000,1.0000
avg_hotel_discount_pct,290,0.0231,0.0619,0.0968,0.1187,0.3192,5046,0.0000,0.0000,0.0000,0.0042,0.4500
avg_hotel_discount_amt,290,6.2000,65.4375,103.6000,160.6500,3150.0000,5112,0.0000,0.0000,0.0000,3.5750,706.5000
total_hotel_disc,290,6.2000,121.3375,261.9500,417.1500,6300.0000,5046,0.0000,0.0000,0.0000,9.1750,1771.2000
total_hotel_only_bookings,290,1.0000,1.0000,1.0000,1.0000,3.0000,5708,0.0000,0.0000,0.0000,1.0000,3.0000


In [20]:
p7_candidates = {
    'hdr>=0.50 AND nights>=5': (
        (df['hotel_discount_rate'] >= 0.50) &
        (df['avg_trip_nights'] >= 5)
    ),
    'hdr>=0.50 AND nights>=6': (
        (df['hotel_discount_rate'] >= 0.50) &
        (df['avg_trip_nights'] >= 6)
    ),
    'hdr>=0.60 AND nights>=5': (
        (df['hotel_discount_rate'] >= 0.60) &
        (df['avg_trip_nights'] >= 5)
    ),
    'hdr>=0.50 AND nights>=5 AND hotel_room>=100': (
        (df['hotel_discount_rate'] >= 0.50) &
        (df['avg_trip_nights'] >= 5) &
        (df['avg_hotel_per_room'] >= 100)
    ),
    'hdr>=0.50 AND disc_amt>=50': (
        (df['hotel_discount_rate'] >= 0.50) &
        (df['avg_hotel_discount_amt'] >= 50)
    ),
}

summary_p7 = compare_candidates(df, p7_candidates, p7_target, 'P7')
summary_p7

  P7 | hdr>=0.50 AND nights>=5
  Target segment :  290 users
  Flagged by rule:  277 users
  ❌ Capture rate  : 140 / 290 = 48.3%   (goal ≥ 75%)
  ✅ False pos rate: 137 / 5,708 = 2.4%   (goal ≤ 15%)

  P7 | hdr>=0.50 AND nights>=6
  Target segment :  290 users
  Flagged by rule:  188 users
  ❌ Capture rate  : 108 / 290 = 37.2%   (goal ≥ 75%)
  ✅ False pos rate: 80 / 5,708 = 1.4%   (goal ≤ 15%)

  P7 | hdr>=0.60 AND nights>=5
  Target segment :  290 users
  Flagged by rule:  135 users
  ❌ Capture rate  : 90 / 290 = 31.0%   (goal ≥ 75%)
  ✅ False pos rate: 45 / 5,708 = 0.8%   (goal ≤ 15%)

  P7 | hdr>=0.50 AND nights>=5 AND hotel_room>=100
  Target segment :  290 users
  Flagged by rule:  230 users
  ❌ Capture rate  : 115 / 290 = 39.7%   (goal ≥ 75%)
  ✅ False pos rate: 115 / 5,708 = 2.0%   (goal ≤ 15%)

  P7 | hdr>=0.50 AND disc_amt>=50
  Target segment :  290 users
  Flagged by rule:  310 users
  ❌ Capture rate  : 174 / 290 = 60.0%   (goal ≥ 75%)
  ✅ False pos rate: 136 / 5,708 = 2.4%  

,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P7 | hdr>=0.50 AND nights>=5,290,277,48.3000,2.4000
P7 | hdr>=0.50 AND nights>=6,290,188,37.2000,1.4000
P7 | hdr>=0.60 AND nights>=5,290,135,31.0000,0.8000
P7 | hdr>=0.50 AND nights>=5 AND hotel_room>=100,290,230,39.7000,2.0000
P7 | hdr>=0.50 AND disc_amt>=50,290,310,60.0000,2.4000


In [21]:
# ── ACCEPTED P7 RULE ───────────────────────────────────────────────────────
P7_RULE = (
    (df['hotel_discount_rate'] >= 0.50) &
    (df['avg_trip_nights'] >= 5)
)
# SQL:
#   hotel_discount_rate >= 0.50
#   AND avg_trip_nights >= 5
#   → Free Hotel Meals

evaluate_rule(df, P7_RULE, p7_target, 'P7 — luxury_hotel_discount_seekers → Free Hotel Meals')

  P7 — luxury_hotel_discount_seekers → Free Hotel Meals
  Target segment :  290 users
  Flagged by rule:  277 users
  ❌ Capture rate  : 140 / 290 = 48.3%   (goal ≥ 75%)
  ✅ False pos rate: 137 / 5,708 = 2.4%   (goal ≤ 15%)



{'label': 'P7 — luxury_hotel_discount_seekers → Free Hotel Meals',
 'n_target': 290,
 'n_flagged': 277,
 'capture_pct': 48.3,
 'fp_pct': 2.4}

### P8 - Free Hotel Meals
**Source dimension:** Path A · spending  
**Target segment:** `hotel_only_domestic_stays` (n≈415)  
**Dimension fields:** Same spending dimension as P4  
**Key signals:** avg_fare_per_seat ~$25 (95.7% NULL), completed_avg_km 163 (very short domestic),
total_hotel_only=1, total_package=0, total_flight_only=0, return_flight_rate=0

In [22]:
P8_DIM_FIELDS = [
    'log_avg_fare_per_seat',
    'log_avg_hotel_per_room',
    'log_completed_total_gross',
    'log_completed_flight_gross',
    'log_completed_hotel_gross',
    # Raw
    'avg_fare_per_seat',
    'avg_hotel_per_room',
    'completed_avg_km',
    'avg_trip_nights',
    'total_hotel_only_bookings',
    'total_package_bookings',
    'total_flight_only_bookings',
    'return_flight_rate',
    'pct_complete_flights_intl',
]

p8_target = df['segment_A_spending'] == 'hotel_only_domestic_stays'
print(f'P8 target segment: {p8_target.sum():,} users')
print()
profile_fields(df[p8_target], df[~p8_target], P8_DIM_FIELDS)

P8 target segment: 415 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
log_avg_fare_per_seat,415,0.0000,0.0000,0.0000,0.0000,3.9012,5583,0.0000,5.3671,5.8040,6.1120,8.0155
log_avg_hotel_per_room,415,3.2189,4.6101,4.9767,5.4293,6.6147,5583,0.0000,4.6898,5.0434,5.3230,6.9698
log_completed_total_gross,415,4.4427,6.5177,7.2306,7.9181,10.0142,5583,0.0000,7.1840,7.8822,8.3907,10.2529
log_completed_flight_gross,415,0.0000,0.0000,0.0000,0.0000,4.5842,5583,0.0000,6.0192,6.7614,7.2546,10.0094
log_completed_hotel_gross,415,4.4427,6.5177,7.2306,7.9181,10.0142,5583,0.0000,6.4536,7.3479,7.9788,10.2477
avg_fare_per_seat,18,5.3500,13.7025,24.3200,30.9675,48.4600,5027,9.2700,252.7608,350.3375,466.7106,3026.4500
avg_hotel_per_room,415,24.0000,99.5000,144.0000,227.0000,745.0000,4921,21.0000,125.5000,164.8000,211.6667,1063.0000
completed_avg_km,18,27.4406,122.5415,163.1911,179.3651,297.4519,5027,47.7445,1468.0615,2015.4739,2660.5153,15813.8407
avg_trip_nights,415,1.0000,4.0000,7.0000,10.0000,30.0000,4921,1.0000,2.6667,3.6667,5.0000,26.0000


In [23]:
# Key insight: hotel_only >= 1, package = 0, very short avg_km (domestic)
# pct_intl = 0 is a strong signal — domestic stays only

p8_candidates = {
    'hotel>=1 AND pkg=0 AND km<=500': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['total_package_bookings'] == 0) &
        (df['completed_avg_km'] <= 500)
    ),
    'hotel>=1 AND pkg=0 AND pct_intl=0': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['total_package_bookings'] == 0) &
        (df['pct_complete_flights_intl'] == 0)
    ),
    'hotel>=1 AND pkg=0 AND km<=300': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['total_package_bookings'] == 0) &
        (df['completed_avg_km'] <= 300)
    ),
    'hotel>=1 AND pkg=0 AND fare<=50': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['total_package_bookings'] == 0) &
        (df['avg_fare_per_seat'] <= 50)
    ),
}

summary_p8 = compare_candidates(df, p8_candidates, p8_target, 'P8')
summary_p8

  P8 | hotel>=1 AND pkg=0 AND km<=500
  Target segment :  415 users
  Flagged by rule:  4 users
  ❌ Capture rate  : 2 / 415 = 0.5%   (goal ≥ 75%)
  ✅ False pos rate: 2 / 5,583 = 0.0%   (goal ≤ 15%)

  P8 | hotel>=1 AND pkg=0 AND pct_intl=0
  Target segment :  415 users
  Flagged by rule:  45 users
  ❌ Capture rate  : 2 / 415 = 0.5%   (goal ≥ 75%)
  ✅ False pos rate: 43 / 5,583 = 0.8%   (goal ≤ 15%)

  P8 | hotel>=1 AND pkg=0 AND km<=300
  Target segment :  415 users
  Flagged by rule:  3 users
  ❌ Capture rate  : 2 / 415 = 0.5%   (goal ≥ 75%)
  ✅ False pos rate: 1 / 5,583 = 0.0%   (goal ≤ 15%)

  P8 | hotel>=1 AND pkg=0 AND fare<=50
  Target segment :  415 users
  Flagged by rule:  2 users
  ❌ Capture rate  : 2 / 415 = 0.5%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,583 = 0.0%   (goal ≤ 15%)



,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P8 | hotel>=1 AND pkg=0 AND km<=500,415,4,0.5000,0.0000
P8 | hotel>=1 AND pkg=0 AND pct_intl=0,415,45,0.5000,0.8000
P8 | hotel>=1 AND pkg=0 AND km<=300,415,3,0.5000,0.0000
P8 | hotel>=1 AND pkg=0 AND fare<=50,415,2,0.5000,0.0000


In [24]:
# ── ACCEPTED P8 RULE ───────────────────────────────────────────────────────
P8_RULE = (
    (df['total_hotel_only_bookings'] >= 1) &
    (df['total_package_bookings'] == 0) &
    (df['completed_avg_km'] <= 500)
)
# SQL:
#   total_hotel_only_bookings >= 1
#   AND total_package_bookings = 0
#   AND completed_avg_km <= 500
#   → Free Hotel Meals

evaluate_rule(df, P8_RULE, p8_target, 'P8 — hotel_only_domestic_stays → Free Hotel Meals')

  P8 — hotel_only_domestic_stays → Free Hotel Meals
  Target segment :  415 users
  Flagged by rule:  4 users
  ❌ Capture rate  : 2 / 415 = 0.5%   (goal ≥ 75%)
  ✅ False pos rate: 2 / 5,583 = 0.0%   (goal ≤ 15%)



{'label': 'P8 — hotel_only_domestic_stays → Free Hotel Meals',
 'n_target': 415,
 'n_flagged': 4,
 'capture_pct': 0.5,
 'fp_pct': 0.0}

### P9 - Exclusive Discounts
**Source dimension:** Path B · flight_discounts  
**Target segment:** `heavy_flight_discount_users` (n≈388)  
**Dimension fields:** Same as P3 flight_discounts dimension  
**Key signals:** flight_discount_rate 0.797, avg_flight_discount_amt $75.92

In [25]:
P9_DIM_FIELDS = [
    'flight_discount_rate',
    'avg_flight_discount_pct',
    'avg_flight_discount_amt',
    'total_flight_disc',
    'overall_discount_pct',
    'discount_dependency',
    'total_flight_only_bookings',
    'total_package_bookings',
]

p9_target = df['segment_B_flight_discounts'] == 'heavy_flight_discount_users'
print(f'P9 target segment: {p9_target.sum():,} users')
print()
profile_fields(df[p9_target], df[~p9_target], P9_DIM_FIELDS)

P9 target segment: 388 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
flight_discount_rate,388,0.2500,0.5000,1.0000,1.0000,1.0000,4657,0.0000,0.0000,0.0000,0.2000,1.0000
avg_flight_discount_pct,388,0.0147,0.0911,0.1092,0.1626,0.4500,4657,0.0000,0.0000,0.0000,0.0074,0.1887
avg_flight_discount_amt,388,0.8710,28.0337,49.1228,74.5095,1862.8275,4792,0.0000,0.0000,0.0000,2.9613,2947.5900
total_flight_disc,388,0.8710,40.7862,79.0672,146.3441,2564.5360,4657,0.0000,0.0000,0.0000,8.6550,389.1420
overall_discount_pct,388,0.0007,0.0332,0.0562,0.0892,0.3500,4657,0.0000,0.0000,0.0000,0.0145,0.2200
discount_dependency,388,0.0000,0.0000,0.0000,0.0000,1.0000,5154,0.0000,0.0000,0.0000,0.0000,1.0000
total_flight_only_bookings,388,0.0000,0.0000,0.0000,1.0000,3.0000,5610,0.0000,0.0000,0.0000,1.0000,3.0000
total_package_bookings,388,0.0000,1.0000,1.0000,2.0000,5.0000,5610,0.0000,1.0000,2.0000,3.0000,8.0000


In [26]:
# Key insight: flight_discount_rate median=0.80 — very high discount dependency
# P3 already took highvolume_package_travelers at fdr>=0.14
# P9 needs a higher threshold to separate heavy discounters from package travelers

p9_candidates = {
    'fdr>=0.50': (df['flight_discount_rate'] >= 0.50),
    'fdr>=0.60': (df['flight_discount_rate'] >= 0.60),
    'fdr>=0.67': (df['flight_discount_rate'] >= 0.67),
    'fdr>=0.50 AND pkg<=1': (
        (df['flight_discount_rate'] >= 0.50) &
        (df['total_package_bookings'] <= 1)
    ),
    'fdr>=0.50 AND disc_amt>=50': (
        (df['flight_discount_rate'] >= 0.50) &
        (df['avg_flight_discount_amt'] >= 50)
    ),
}

summary_p9 = compare_candidates(df, p9_candidates, p9_target, 'P9')
summary_p9

  P9 | fdr>=0.50
  Target segment :  388 users
  Flagged by rule:  745 users
  ✅ Capture rate  : 366 / 388 = 94.3%   (goal ≥ 75%)
  ✅ False pos rate: 379 / 5,610 = 6.8%   (goal ≤ 15%)

  P9 | fdr>=0.60
  Target segment :  388 users
  Flagged by rule:  298 users
  ❌ Capture rate  : 259 / 388 = 66.8%   (goal ≥ 75%)
  ✅ False pos rate: 39 / 5,610 = 0.7%   (goal ≤ 15%)

  P9 | fdr>=0.67
  Target segment :  388 users
  Flagged by rule:  232 users
  ❌ Capture rate  : 228 / 388 = 58.8%   (goal ≥ 75%)
  ✅ False pos rate: 4 / 5,610 = 0.1%   (goal ≤ 15%)

  P9 | fdr>=0.50 AND pkg<=1
  Target segment :  388 users
  Flagged by rule:  277 users
  ❌ Capture rate  : 226 / 388 = 58.2%   (goal ≥ 75%)
  ✅ False pos rate: 51 / 5,610 = 0.9%   (goal ≤ 15%)

  P9 | fdr>=0.50 AND disc_amt>=50
  Target segment :  388 users
  Flagged by rule:  185 users
  ❌ Capture rate  : 170 / 388 = 43.8%   (goal ≥ 75%)
  ✅ False pos rate: 15 / 5,610 = 0.3%   (goal ≤ 15%)



,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P9 | fdr>=0.50,388,745,94.3000,6.8000
P9 | fdr>=0.60,388,298,66.8000,0.7000
P9 | fdr>=0.67,388,232,58.8000,0.1000
P9 | fdr>=0.50 AND pkg<=1,388,277,58.2000,0.9000
P9 | fdr>=0.50 AND disc_amt>=50,388,185,43.8000,0.3000


In [27]:
# ── ACCEPTED P9 RULE ───────────────────────────────────────────────────────
P9_RULE = (df['flight_discount_rate'] >= 0.50)
# SQL: flight_discount_rate >= 0.50  → Exclusive Discounts

evaluate_rule(df, P9_RULE, p9_target, 'P9 — heavy_flight_discount_users → Exclusive Discounts')

  P9 — heavy_flight_discount_users → Exclusive Discounts
  Target segment :  388 users
  Flagged by rule:  745 users
  ✅ Capture rate  : 366 / 388 = 94.3%   (goal ≥ 75%)
  ✅ False pos rate: 379 / 5,610 = 6.8%   (goal ≤ 15%)



{'label': 'P9 — heavy_flight_discount_users → Exclusive Discounts',
 'n_target': 388,
 'n_flagged': 745,
 'capture_pct': 94.3,
 'fp_pct': 6.8}

### P10 - Free Hotel Meals
**Source dimension:** Path A · travel  
**Target segment:** `hotel_centric_short_haul` (n≈401)  
**Dimension fields:** `avg_trip_nights`, `avg_checked_bags`, `avg_seats`, `return_flight_rate`,
`destination_variety`, `airline_variety`, `lead_time_bin_enc`, `log_avg_lead_time_days`,
`log_completed_avg_km`, `pct_complete_flights_intl`, `pct_complete_flights_domestic`,
`total_package_bookings`, `total_flight_only_bookings`, `total_hotel_only_bookings`  
**Key signals:** return_flight_rate≈0 (near zero), destination_variety≈0 (near zero), avg_trip_nights 7.8

In [ ]:
P10_DIM_FIELDS = [
    'avg_trip_nights',
    'avg_checked_bags',
    'avg_seats',
    'return_flight_rate',
    'destination_variety',
    'airline_variety',
    'lead_time_bin_enc',
    'log_avg_lead_time_days',
    'log_completed_avg_km',
    'pct_complete_flights_intl',
    'pct_complete_flights_domestic',
    'total_package_bookings',
    'total_flight_only_bookings',
    'total_hotel_only_bookings',
    # Raw
    'completed_avg_km',
]

p10_target = df['segment_A_travel'] == 'hotel_centric_short_haul'
print(f'P10 target segment: {p10_target.sum():,} users')
print()
profile_fields(df[p10_target], df[~p10_target], P10_DIM_FIELDS)

In [ ]:
p10_candidates = {
    'nights>=6 AND hotel>=1 AND pkg<=1': (
        (df['avg_trip_nights'] >= 6) &
        (df['total_hotel_only_bookings'] >= 1) &
        (df['total_package_bookings'] <= 1)
    ),
    'nights>=7 AND hotel>=1': (
        (df['avg_trip_nights'] >= 7) &
        (df['total_hotel_only_bookings'] >= 1)
    ),
    'nights>=6 AND return_rate<=0.10': (
        (df['avg_trip_nights'] >= 6) &
        (df['return_flight_rate'] <= 0.10)
    ),
    'nights>=7 AND hotel>=1 AND km<=1000': (
        (df['avg_trip_nights'] >= 7) &
        (df['total_hotel_only_bookings'] >= 1) &
        (df['completed_avg_km'] <= 1000)
    ),
}

summary_p10 = compare_candidates(df, p10_candidates, p10_target, 'P10')
summary_p10

In [ ]:
# ── ACCEPTED P10 RULE ──────────────────────────────────────────────────────
P10_RULE = (
    (df['avg_trip_nights'] >= 6) &
    (df['total_hotel_only_bookings'] >= 1) &
    (df['total_package_bookings'] <= 1)
)
# SQL:
#   avg_trip_nights >= 6
#   AND total_hotel_only_bookings >= 1
#   AND total_package_bookings <= 1
#   → Free Hotel Meals

evaluate_rule(df, P10_RULE, p10_target, 'P10 — hotel_centric_short_haul → Free Hotel Meals')

### P11 - No Change Fees
**Source dimension:** Path A · cancellation  
**Target segment:** `highvalue_advanced_cancellers` (n≈79)  
**Dimension fields:** `total_cancellation_rate`, `cancel_spend_ratio`, `log_cxl_gross_spend`,
`log_avg_cancelled_lead_time`, `cancelled_lead_time_bin_enc`, `log_avg_net_cancelled_flight`,
`log_avg_net_cancelled_hotel`, `hotel_only_cxl_rate`, `flight_only_cxl_rate`, `package_cxl_rate`  
**Key signals:** avg_cancelled_lead_time p10=102 (ALL > 100 days), cancel_spend_ratio 0.678, package_cxl_rate 0.50

In [ ]:
P11_DIM_FIELDS = [
    'total_cancellation_rate',
    'cancel_spend_ratio',
    'log_cxl_gross_spend',
    'log_avg_cancelled_lead_time',
    'cancelled_lead_time_bin_enc',
    'log_avg_net_cancelled_flight',
    'log_avg_net_cancelled_hotel',
    'hotel_only_cxl_rate',
    'flight_only_cxl_rate',
    'package_cxl_rate',
    # Raw
    'avg_cancelled_lead_time',
    'cxl_gross_spend',
]

p11_target = df['segment_A_cancellation'] == 'highvalue_advanced_cancellers'
print(f'P11 target segment: {p11_target.sum():,} users')
print()
profile_fields(df[p11_target], df[~p11_target], P11_DIM_FIELDS)

In [ ]:
# Key insight: ALL members have avg_cancelled_lead_time > 100 days
# cancel_spend_ratio is the primary value signal
# package_cxl_rate p50=0.50 — package cancellers specifically

p11_candidates = {
    'spend_ratio>=0.50 AND lead_time>=100': (
        (df['cancel_spend_ratio'] >= 0.50) &
        (df['avg_cancelled_lead_time'] >= 100)
    ),
    'cxl_rate>=0.25 AND spend_ratio>=0.50': (
        (df['total_cancellation_rate'] >= 0.25) &
        (df['cancel_spend_ratio'] >= 0.50)
    ),
    'spend_ratio>=0.60 AND lead_time>=90': (
        (df['cancel_spend_ratio'] >= 0.60) &
        (df['avg_cancelled_lead_time'] >= 90)
    ),
    'pkg_cxl>=0.33 AND spend_ratio>=0.50': (
        (df['package_cxl_rate'] >= 0.33) &
        (df['cancel_spend_ratio'] >= 0.50)
    ),
    'spend_ratio>=0.50 AND lead_time>=100 AND pkg_cxl>=0.25': (
        (df['cancel_spend_ratio'] >= 0.50) &
        (df['avg_cancelled_lead_time'] >= 100) &
        (df['package_cxl_rate'] >= 0.25)
    ),
}

summary_p11 = compare_candidates(df, p11_candidates, p11_target, 'P11')
summary_p11

In [ ]:
# ── ACCEPTED P11 RULE ──────────────────────────────────────────────────────
P11_RULE = (
    (df['cancel_spend_ratio'] >= 0.50) &
    (df['avg_cancelled_lead_time'] >= 100)
)
# SQL:
#   cancel_spend_ratio >= 0.50
#   AND avg_cancelled_lead_time >= 100
#   → No Change Fees

evaluate_rule(df, P11_RULE, p11_target, 'P11 — highvalue_advanced_cancellers → No Change Fees')

### P12 - Free Hotel Meals
**Source dimension:** Path B · hotel_behavior  
**Target segment:** `regular_hotel_bookers` (n≈1,698)  
**Dimension fields:** Same hotel_behavior dimension as P7  
**Key signals:** total_hotel_only_bookings=1 (constant min), avg_trip_nights 5.6, completed_hotel_gross $2,520 (p50)

In [ ]:
P12_DIM_FIELDS = [
    'avg_trip_nights',
    'log_avg_hotel_total_per_stay',
    'log_avg_hotel_per_room',
    'log_completed_hotel_gross',
    'hotel_discount_rate',
    'avg_hotel_discount_pct',
    'avg_hotel_discount_amt',
    'total_hotel_disc',
    'total_hotel_only_bookings',
    # Raw
    'avg_hotel_per_room',
    'completed_hotel_gross',
    'total_package_bookings',
]

p12_target = df['segment_B_hotel_behavior'] == 'regular_hotel_bookers'
print(f'P12 target segment: {p12_target.sum():,} users')
print()
profile_fields(df[p12_target], df[~p12_target], P12_DIM_FIELDS)

In [ ]:
p12_candidates = {
    'hotel>=1 AND nights>=4': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['avg_trip_nights'] >= 4)
    ),
    'hotel>=1 AND nights>=3 AND room>0': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['avg_trip_nights'] >= 3) &
        (df['avg_hotel_per_room'] > 0)
    ),
    'hotel>=1 AND nights>=5': (
        (df['total_hotel_only_bookings'] >= 1) &
        (df['avg_trip_nights'] >= 5)
    ),
    'hotel>=1': (df['total_hotel_only_bookings'] >= 1),
}

summary_p12 = compare_candidates(df, p12_candidates, p12_target, 'P12')
summary_p12

In [ ]:
# ── ACCEPTED P12 RULE ──────────────────────────────────────────────────────
P12_RULE = (
    (df['total_hotel_only_bookings'] >= 1) &
    (df['avg_trip_nights'] >= 4)
)
# SQL:
#   total_hotel_only_bookings >= 1
#   AND avg_trip_nights >= 4
#   → Free Hotel Meals

evaluate_rule(df, P12_RULE, p12_target, 'P12 — regular_hotel_bookers → Free Hotel Meals')

### P13 - Premium Travel Benefits
**Source dimension:** Path B · intl_travel  
**Target segment:** `frequent_mixed_travelers` (n≈2,009)  
**Dimension fields:** Same intl_travel dimension as P5  
**Key signals:** pct_intl 0.44, trips_per_year 3.8 (p10=1.5), total_package_bookings 3.0, completed_total_gross $4,270

In [ ]:
P13_DIM_FIELDS = [
    'pct_complete_flights_intl',
    'pct_complete_flights_domestic',
    'log_completed_avg_km',
    'log_attempted_avg_km',
    'cxl_avg_km',
    'pct_cxl_flights_intl',
    'avg_checked_bags',
    'avg_seats',
    'return_flight_rate',
    'total_package_bookings',
    # Raw
    'trips_per_year',
    'completed_total_gross',
    'completed_avg_km',
]

p13_target = df['segment_B_intl_travel'] == 'frequent_mixed_travelers'
print(f'P13 target segment: {p13_target.sum():,} users')
print()
profile_fields(df[p13_target], df[~p13_target], P13_DIM_FIELDS)

In [ ]:
p13_candidates = {
    'pkg>=2 AND trips>=1.5': (
        (df['total_package_bookings'] >= 2) &
        (df['trips_per_year'] >= 1.5)
    ),
    'pkg>=2 AND gross>=2500': (
        (df['total_package_bookings'] >= 2) &
        (df['completed_total_gross'] >= 2500)
    ),
    'pkg>=3 AND trips>=1.5': (
        (df['total_package_bookings'] >= 3) &
        (df['trips_per_year'] >= 1.5)
    ),
    'trips>=1.5 AND gross>=3000': (
        (df['trips_per_year'] >= 1.5) &
        (df['completed_total_gross'] >= 3000)
    ),
    'pkg>=2 AND trips>=2': (
        (df['total_package_bookings'] >= 2) &
        (df['trips_per_year'] >= 2)
    ),
}

summary_p13 = compare_candidates(df, p13_candidates, p13_target, 'P13')
summary_p13

In [ ]:
# ── ACCEPTED P13 RULE ──────────────────────────────────────────────────────
P13_RULE = (
    (df['total_package_bookings'] >= 2) &
    (df['trips_per_year'] >= 1.5)
)
# SQL:
#   total_package_bookings >= 2
#   AND trips_per_year >= 1.5
#   → Premium Travel Benefits

evaluate_rule(df, P13_RULE, p13_target, 'P13 — frequent_mixed_travelers → Premium Travel Benefits')

### P14 - No Change Fees
**Source dimension:** Path A · cancellation  
**Target segment:** `flight_advanced_planner_cancellers` (n≈140)  
**Dimension fields:** Same cancellation dimension as P11  
**Key signals:** flight_only_cxl_rate=1.0 (constant), hotel_only_cxl_rate=0.0, package_cxl_rate=0.0, avg_cancelled_lead_time p10=9, p25=196

In [ ]:
P14_DIM_FIELDS = [
    'total_cancellation_rate',
    'cancel_spend_ratio',
    'flight_only_cxl_rate',
    'hotel_only_cxl_rate',
    'package_cxl_rate',
    'avg_cancelled_lead_time',
    'log_avg_cancelled_lead_time',
    'cancelled_lead_time_bin_enc',
    'log_avg_net_cancelled_flight',
    'cxl_gross_spend',
    # Raw supporting
    'total_hotel_only_bookings',
    'total_package_bookings',
]

p14_target = df['segment_A_cancellation'] == 'flight_advanced_planner_cancellers'
print(f'P14 target segment: {p14_target.sum():,} users')
print()
profile_fields(df[p14_target], df[~p14_target], P14_DIM_FIELDS)

In [ ]:
# Key insight: flight_only_cxl_rate = 1.0 is constant — pure flight cancellers
# hotel_only_cxl_rate = 0, package_cxl_rate = 0 (these users only cancel flights)

p14_candidates = {
    'flight_cxl=1.0 AND hotel_cxl=0 AND pkg_cxl=0': (
        (df['flight_only_cxl_rate'] == 1.0) &
        (df['hotel_only_cxl_rate'] == 0) &
        (df['package_cxl_rate'] == 0)
    ),
    'flight_cxl>=0.80 AND hotel_cxl=0': (
        (df['flight_only_cxl_rate'] >= 0.80) &
        (df['hotel_only_cxl_rate'] == 0)
    ),
    'flight_cxl=1.0 AND hotel_only=0 AND pkg=0': (
        (df['flight_only_cxl_rate'] == 1.0) &
        (df['total_hotel_only_bookings'] == 0) &
        (df['total_package_bookings'] == 0)
    ),
    'cxl_rate>0 AND hotel_only=0 AND pkg=0': (
        (df['total_cancellation_rate'] > 0) &
        (df['total_hotel_only_bookings'] == 0) &
        (df['total_package_bookings'] == 0)
    ),
}

summary_p14 = compare_candidates(df, p14_candidates, p14_target, 'P14')
summary_p14

In [ ]:
# ── ACCEPTED P14 RULE ──────────────────────────────────────────────────────
P14_RULE = (
    (df['flight_only_cxl_rate'] == 1.0) &
    (df['hotel_only_cxl_rate'] == 0) &
    (df['package_cxl_rate'] == 0)
)
# SQL:
#   flight_only_cxl_rate = 1.0
#   AND hotel_only_cxl_rate = 0
#   AND package_cxl_rate = 0
#   → No Change Fees

evaluate_rule(df, P14_RULE, p14_target, 'P14 — flight_advanced_planner_cancellers → No Change Fees')

### P15 - No Change Fees
**Source dimension:** Path A · cancellation  
**Target segment:** `package_cancellers` (n≈232)  
**Dimension fields:** Same cancellation dimension as P11/P14  
**Key signals:** package_cxl_rate 0.333 (p50), avg_cancelled_lead_time p50=7 (short notice), hotel_only_cxl_rate=0.0

In [ ]:
P15_DIM_FIELDS = [
    'total_cancellation_rate',
    'cancel_spend_ratio',
    'flight_only_cxl_rate',
    'hotel_only_cxl_rate',
    'package_cxl_rate',
    'avg_cancelled_lead_time',
    'log_avg_cancelled_lead_time',
    'cancelled_lead_time_bin_enc',
    'cxl_gross_spend',
    # Supporting
    'total_package_bookings',
    'total_hotel_only_bookings',
]

p15_target = df['segment_A_cancellation'] == 'package_cancellers'
print(f'P15 target segment: {p15_target.sum():,} users')
print()
profile_fields(df[p15_target], df[~p15_target], P15_DIM_FIELDS)

In [ ]:
# Key insight: package cancellers — have packages AND cancellations, hotel_only=0
# package_cxl_rate > 0 is the defining signal

p15_candidates = {
    'pkg_cxl>0 AND hotel_cxl=0': (
        (df['package_cxl_rate'] > 0) &
        (df['hotel_only_cxl_rate'] == 0)
    ),
    'pkg_cxl>=0.25 AND hotel_cxl=0': (
        (df['package_cxl_rate'] >= 0.25) &
        (df['hotel_only_cxl_rate'] == 0)
    ),
    'pkg_cxl>0 AND hotel_only=0': (
        (df['package_cxl_rate'] > 0) &
        (df['total_hotel_only_bookings'] == 0)
    ),
    'cxl_rate>0 AND pkg>=1 AND hotel_only=0': (
        (df['total_cancellation_rate'] > 0) &
        (df['total_package_bookings'] >= 1) &
        (df['total_hotel_only_bookings'] == 0)
    ),
}

summary_p15 = compare_candidates(df, p15_candidates, p15_target, 'P15')
summary_p15

In [ ]:
# ── ACCEPTED P15 RULE ──────────────────────────────────────────────────────
P15_RULE = (
    (df['package_cxl_rate'] > 0) &
    (df['hotel_only_cxl_rate'] == 0)
)
# SQL:
#   package_cxl_rate > 0
#   AND hotel_only_cxl_rate = 0
#   → No Change Fees

evaluate_rule(df, P15_RULE, p15_target, 'P15 — package_cancellers → No Change Fees')

### P16 - No Change Fees
**Source dimension:** Path B · intl_travel  
**Target segment:** `intl_trip_cancellers` (n≈130)  
**Dimension fields:** Same intl_travel dimension as P5/P13  
**Key signals:** pct_cxl_flights_intl=1.0 (constant), total_cancellation_rate 0.333, avg_cancelled_lead_time p50=169

In [ ]:
P16_DIM_FIELDS = [
    'pct_complete_flights_intl',
    'pct_cxl_flights_intl',
    'pct_cxl_flights_domestic',
    'total_cancellation_rate',
    'cancel_spend_ratio',
    'avg_cancelled_lead_time',
    'cxl_avg_km',
    'completed_avg_km',
    # Supporting
    'total_package_bookings',
    'hotel_only_cxl_rate',
    'flight_only_cxl_rate',
    'package_cxl_rate',
]

p16_target = df['segment_B_intl_travel'] == 'intl_trip_cancellers'
print(f'P16 target segment: {p16_target.sum():,} users')
print()
profile_fields(df[p16_target], df[~p16_target], P16_DIM_FIELDS)

In [ ]:
# Key insight: pct_cxl_flights_intl = 1.0 is constant — all cancellations were intl flights
# Combined with meaningful cancellation rate

p16_candidates = {
    'pct_cxl_intl=1.0 AND cxl_rate>=0.20': (
        (df['pct_cxl_flights_intl'] == 1.0) &
        (df['total_cancellation_rate'] >= 0.20)
    ),
    'pct_cxl_intl=1.0 AND cxl_rate>=0.25': (
        (df['pct_cxl_flights_intl'] == 1.0) &
        (df['total_cancellation_rate'] >= 0.25)
    ),
    'pct_intl>=0.50 AND cxl_rate>=0.25 AND km>=1500': (
        (df['pct_complete_flights_intl'] >= 0.50) &
        (df['total_cancellation_rate'] >= 0.25) &
        (df['completed_avg_km'] >= 1500)
    ),
    'pct_cxl_intl=1.0 AND cxl_rate>=0.20 AND lead>=90': (
        (df['pct_cxl_flights_intl'] == 1.0) &
        (df['total_cancellation_rate'] >= 0.20) &
        (df['avg_cancelled_lead_time'] >= 90)
    ),
}

summary_p16 = compare_candidates(df, p16_candidates, p16_target, 'P16')
summary_p16

In [ ]:
# ── ACCEPTED P16 RULE ──────────────────────────────────────────────────────
P16_RULE = (
    (df['pct_cxl_flights_intl'] == 1.0) &
    (df['total_cancellation_rate'] >= 0.20)
)
# SQL:
#   pct_cxl_flights_intl = 1.0
#   AND total_cancellation_rate >= 0.20
#   → No Change Fees

evaluate_rule(df, P16_RULE, p16_target, 'P16 — intl_trip_cancellers → No Change Fees')

### P17 - No Change Fees
**Source dimension:** Path A · cancellation  
**Target segment:** `hotel_lastminute_cancellers` (n≈44)  
**Dimension fields:** Same cancellation dimension as P11/P14/P15  
**Key signals:** hotel_only_cxl_rate >= 0.50, avg_cancelled_lead_time p50=9 (short notice)

In [28]:
P17_DIM_FIELDS = [
    'total_cancellation_rate',
    'cancel_spend_ratio',
    'hotel_only_cxl_rate',
    'flight_only_cxl_rate',
    'package_cxl_rate',
    'avg_cancelled_lead_time',
    'cancelled_lead_time_bin_enc',
    'log_avg_net_cancelled_hotel',
    'cxl_hotel_gross',
    # Supporting
    'total_hotel_only_bookings',
    'total_package_bookings',
]

p17_target = df['segment_A_cancellation'] == 'hotel_lastminute_cancellers'
print(f'P17 target segment: {p17_target.sum():,} users')
print()
profile_fields(df[p17_target], df[~p17_target], P17_DIM_FIELDS)

P17 target segment: 44 users



,TARGET_n_nonull,TARGET_min,TARGET_p25,TARGET_p50,TARGET_p75,TARGET_max,OTHERS_n_nonull,OTHERS_min,OTHERS_p25,OTHERS_p50,OTHERS_p75,OTHERS_max
field,,,,,,,,,,,,
total_cancellation_rate,44,0.2000,0.2500,0.3333,0.5000,0.6667,5954,0.0000,0.0000,0.0000,0.0000,1.0000
cancel_spend_ratio,44,0.0392,0.2039,0.2988,0.4530,0.9586,5954,0.0000,0.0000,0.0000,0.0000,1.0000
hotel_only_cxl_rate,44,0.3333,0.5000,1.0000,1.0000,1.0000,5954,0.0000,0.0000,0.0000,0.0000,0.5000
flight_only_cxl_rate,44,0.0000,0.0000,0.0000,0.0000,1.0000,5954,0.0000,0.0000,0.0000,0.0000,1.0000
package_cxl_rate,44,0.0000,0.0000,0.0000,0.0000,0.5000,5954,0.0000,0.0000,0.0000,0.0000,1.0000
avg_cancelled_lead_time,44,1.0000,5.7500,9.0000,16.0000,155.5000,5954,0.0000,0.0000,0.0000,0.0000,364.0000
cancelled_lead_time_bin_enc,44,0.0000,0.0000,1.0000,2.0000,3.0000,5954,-1.0000,-1.0000,-1.0000,-1.0000,3.0000
log_avg_net_cancelled_hotel,44,5.3288,6.3052,6.8056,7.3457,9.1919,5954,0.0000,0.0000,0.0000,0.0000,9.2569
cxl_hotel_gross,44,228.0000,561.2500,1111.0000,1692.5000,9816.0000,5954,0.0000,0.0000,0.0000,0.0000,10476.0000


In [29]:
# Key insight: hotel_only_cxl_rate is the defining signal — pure hotel cancellers
# Short lead times (p50=9 days) distinguish from highvalue_advanced_cancellers

p17_candidates = {
    'hotel_cxl>=0.50 AND pkg_cxl=0': (
        (df['hotel_only_cxl_rate'] >= 0.50) &
        (df['package_cxl_rate'] == 0)
    ),
    'hotel_cxl>0 AND pkg_cxl=0 AND flight_cxl=0': (
        (df['hotel_only_cxl_rate'] > 0) &
        (df['package_cxl_rate'] == 0) &
        (df['flight_only_cxl_rate'] == 0)
    ),
    'hotel_cxl>=0.33 AND pkg_cxl=0': (
        (df['hotel_only_cxl_rate'] >= 0.33) &
        (df['package_cxl_rate'] == 0)
    ),
    'hotel_cxl>=0.50 AND lead<=30': (
        (df['hotel_only_cxl_rate'] >= 0.50) &
        (df['avg_cancelled_lead_time'] <= 30)
    ),
}

summary_p17 = compare_candidates(df, p17_candidates, p17_target, 'P17')
summary_p17

  P17 | hotel_cxl>=0.50 AND pkg_cxl=0
  Target segment :  44 users
  Flagged by rule:  40 users
  ✅ Capture rate  : 40 / 44 = 90.9%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,954 = 0.0%   (goal ≤ 15%)

  P17 | hotel_cxl>0 AND pkg_cxl=0 AND flight_cxl=0
  Target segment :  44 users
  Flagged by rule:  40 users
  ✅ Capture rate  : 40 / 44 = 90.9%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,954 = 0.0%   (goal ≤ 15%)

  P17 | hotel_cxl>=0.33 AND pkg_cxl=0
  Target segment :  44 users
  Flagged by rule:  42 users
  ✅ Capture rate  : 42 / 44 = 95.5%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,954 = 0.0%   (goal ≤ 15%)

  P17 | hotel_cxl>=0.50 AND lead<=30
  Target segment :  44 users
  Flagged by rule:  38 users
  ✅ Capture rate  : 38 / 44 = 86.4%   (goal ≥ 75%)
  ✅ False pos rate: 0 / 5,954 = 0.0%   (goal ≤ 15%)



,n_target,n_flagged,capture_pct,fp_pct
label,,,,
P17 | hotel_cxl>=0.50 AND pkg_cxl=0,44,40,90.9000,0.0000
P17 | hotel_cxl>0 AND pkg_cxl=0 AND flight_cxl=0,44,40,90.9000,0.0000
P17 | hotel_cxl>=0.33 AND pkg_cxl=0,44,42,95.5000,0.0000
P17 | hotel_cxl>=0.50 AND lead<=30,44,38,86.4000,0.0000


In [ ]:
# ── ACCEPTED P17 RULE ──────────────────────────────────────────────────────
P17_RULE = (
    (df['hotel_only_cxl_rate'] >= 0.50) &
    (df['package_cxl_rate'] == 0)
)
# SQL:
#   hotel_only_cxl_rate >= 0.50
#   AND package_cxl_rate = 0
#   → No Change Fees

evaluate_rule(df, P17_RULE, p17_target, 'P17 — hotel_lastminute_cancellers → No Change Fees')

### Full Sequential Assignment - Final Distribution
Apply all rules in priority order. Each user gets exactly one perk (first matching rule wins).

In [ ]:
assigned = pd.Series('UNASSIGNED', index=df.index)

# P1 — never-booked → Exclusive Discounts
p1 = df['user_booking_status'] == 'never_booked'
assigned[p1] = 'Exclusive Discounts'

# P2 — cancelled-all → No Change Fees
p2 = (df['user_booking_status'] == 'cancelled_all') & (assigned == 'UNASSIGNED')
assigned[p2] = 'No Change Fees'

# P3–P17: sequential (first match wins)
rules = [
    ('P3',  P3_RULE,  'Premium Travel Benefits'),
    ('P4',  P4_RULE,  'Premium Travel Benefits'),
    ('P5',  P5_RULE,  'Free Checked Bags'),
    ('P7',  P7_RULE,  'Free Hotel Meals'),
    ('P8',  P8_RULE,  'Free Hotel Meals'),
    ('P9',  P9_RULE,  'Exclusive Discounts'),
    ('P10', P10_RULE, 'Free Hotel Meals'),
    ('P11', P11_RULE, 'No Change Fees'),
    ('P12', P12_RULE, 'Free Hotel Meals'),
    ('P13', P13_RULE, 'Premium Travel Benefits'),
    ('P14', P14_RULE, 'No Change Fees'),
    ('P15', P15_RULE, 'No Change Fees'),
    ('P16', P16_RULE, 'No Change Fees'),
    ('P17', P17_RULE, 'No Change Fees'),
]

for priority, rule, perk in rules:
    eligible = rule & (assigned == 'UNASSIGNED')
    assigned[eligible] = perk
    print(f"{priority}: assigned {eligible.sum():>5,} users → {perk}")

# P18 — high-gross fallback
p18 = (df['completed_total_gross'] >= 3000) & (assigned == 'UNASSIGNED')
assigned[p18] = 'Premium Travel Benefits'
print(f"P18: assigned {p18.sum():>5,} users → Premium Travel Benefits (fallback)")

# P19 — default fallback
p19 = assigned == 'UNASSIGNED'
assigned[p19] = 'Exclusive Discounts'
print(f"P19: assigned {p19.sum():>5,} users → Exclusive Discounts (default)")

In [ ]:
# ── Final distribution vs segment-based targets ────────────────────────────
targets = {
    'Premium Travel Benefits': 2438,
    'Exclusive Discounts':     1269,
    'Free Hotel Meals':        1249,
    'Free Checked Bags':        807,
    'No Change Fees':           235,
}

dist = assigned.value_counts()
print(f"\n{'Perk':<30} {'Actual':>8} {'Target':>8} {'Delta':>8} {'Status'}")
print('-' * 65)
for perk, target_n in targets.items():
    actual_n = dist.get(perk, 0)
    delta    = actual_n - target_n
    flag     = '✅' if abs(delta) <= 200 else '⚠️ '
    print(f"{flag} {perk:<28} {actual_n:>8,} {target_n:>8,} {delta:>+8,}")

total = len(df)
print(f"\nTotal assigned: {dist.sum():,} / {total:,}")

In [ ]:
# ── Overall rule summary table ─────────────────────────────────────────────
all_results = []
rule_defs = [
    ('P3',  P3_RULE,  p3_target,  'highvolume_package_travelers',        'Premium Travel Benefits'),
    ('P4',  P4_RULE,  p4_target,  'flight_only_high_fare',               'Premium Travel Benefits'),
    ('P5',  P5_RULE,  p5_target,  'dedicated_intl_travelers',            'Free Checked Bags'),
    ('P7',  P7_RULE,  p7_target,  'luxury_hotel_discount_seekers',       'Free Hotel Meals'),
    ('P8',  P8_RULE,  p8_target,  'hotel_only_domestic_stays',           'Free Hotel Meals'),
    ('P9',  P9_RULE,  p9_target,  'heavy_flight_discount_users',         'Exclusive Discounts'),
    ('P10', P10_RULE, p10_target, 'hotel_centric_short_haul',            'Free Hotel Meals'),
    ('P11', P11_RULE, p11_target, 'highvalue_advanced_cancellers',       'No Change Fees'),
    ('P12', P12_RULE, p12_target, 'regular_hotel_bookers',               'Free Hotel Meals'),
    ('P13', P13_RULE, p13_target, 'frequent_mixed_travelers',            'Premium Travel Benefits'),
    ('P14', P14_RULE, p14_target, 'flight_advanced_planner_cancellers',  'No Change Fees'),
    ('P15', P15_RULE, p15_target, 'package_cancellers',                  'No Change Fees'),
    ('P16', P16_RULE, p16_target, 'intl_trip_cancellers',                'No Change Fees'),
    ('P17', P17_RULE, p17_target, 'hotel_lastminute_cancellers',         'No Change Fees'),
]

rows = []
for priority, rule, target_mask, segment, perk in rule_defs:
    n_t   = int(target_mask.sum())
    n_o   = int((~target_mask).sum())
    n_cap = int((rule & target_mask).sum())
    n_fp  = int((rule & ~target_mask).sum())
    cap   = round(n_cap / n_t * 100, 1) if n_t else 0
    fp    = round(n_fp / n_o * 100, 1) if n_o else 0
    ok    = '✅' if cap >= 75 and fp <= 15 else '❌'
    rows.append({'P': priority, 'Segment': segment, 'Perk': perk,
                 'n_target': n_t, 'n_flagged': n_cap + n_fp,
                 'capture_%': cap, 'fp_%': fp, 'pass': ok})

summary_all = pd.DataFrame(rows).set_index('P')
summary_all